In [3]:
# Cell 1: Setup & Configuration
import logging
import statistics
import time

import pandas as pd
import requests
from IPython.display import display

# Configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)

# Local model endpoint (MLX / llama.cpp on MacBook)
MODEL_NAME = "mlx-community/gemma-4-e4b-it-4bit"
LOCAL_BASE = "http://localhost:8000"
LOCAL_CHAT_URL = f"{LOCAL_BASE}/v1/chat/completions"

logger.info(f"Local model: {MODEL_NAME}")
logger.info(f"Chat endpoint: {LOCAL_CHAT_URL}")

headers = {"Content-Type": "application/json"}

# Same benchmark prompts used in the Spark Cluster test
NUM_REQUESTS = 5
BENCHMARK_PROMPTS = [
    "What is the capital of France?",
    "Write a short poem about artificial intelligence.",
    "Explain quantum computing in simple terms.",
    "What are the three laws of robotics?",
    "Summarize the plot of the movie Inception.",
]


2026-10-06 01:26:58,953 - INFO - Local model: mlx-community/gemma-4-e4b-it-4bit
2026-10-06 01:26:58,955 - INFO - Chat endpoint: http://localhost:8000/v1/chat/completions


In [7]:
# Cell 2: Benchmark & Comparison
print("=" * 60)
print("GEMMA 4 (LOCAL) - PERFORMANCE BENCHMARK")
print(f"Model: {MODEL_NAME}")
print(f"Endpoint: {LOCAL_CHAT_URL}")
print("=" * 60)

latencies = []
responses_collected = []

print(f"\nRunning benchmark: {NUM_REQUESTS} requests...")
print("-" * 60)

for i in range(NUM_REQUESTS):
    prompt = BENCHMARK_PROMPTS[i % len(BENCHMARK_PROMPTS)]

    # Chat completions payload (OpenAI-compatible format)
    bench_payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 150,
        "temperature": 0.7,
    }

    try:
        start = time.time()

        response = requests.post(
            LOCAL_CHAT_URL,
            headers=headers,
            json=bench_payload,
            timeout=120,
        )

        elapsed = time.time() - start
        latencies.append(elapsed)

        data = response.json()
        generated_text = (
            data.get("choices", [{}])[0].get("message", {}).get("content", "")
        )

        # Token usage
        usage = data.get("usage", {})
        prompt_tokens = usage.get("prompt_tokens", 0)
        completion_tokens = usage.get("completion_tokens", 0)
        tokens_per_second = (
            round(completion_tokens / elapsed, 1)
            if elapsed > 0 and completion_tokens
            else 0
        )

        responses_collected.append(
            {
                "request": i + 1,
                "prompt": prompt[:50] + "..." if len(prompt) > 50 else prompt,
                "latency_s": round(elapsed, 3),
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "tokens/s": tokens_per_second,
                "status": response.status_code,
                "generated_text": generated_text,
            }
        )

        print(
            f"Request {i + 1:2d}/{NUM_REQUESTS}: {elapsed:.3f}s "
            f"- Status {response.status_code} "
            f"({completion_tokens} tok, {tokens_per_second} tok/s)"
        )

    except (requests.exceptions.RequestException, ValueError, OSError) as e:
        logger.error(f"Request {i + 1} failed: {e}")
        latencies.append(float("inf"))
        responses_collected.append(
            {
                "request": i + 1,
                "prompt": prompt[:50],
                "latency_s": float("inf"),
                "prompt_tokens": 0,
                "completion_tokens": 0,
                "tokens/s": 0,
                "status": "FAILED",
            }
        )

# -----------------------------------------------------------------------
# Results Summary
# -----------------------------------------------------------------------
print("-" * 60)
print("\nLOCAL BENCHMARK RESULTS")
print("=" * 60)

valid_latencies = [v for v in latencies if v != float("inf")]
valid_responses = [r for r in responses_collected if r["status"] != "FAILED"]

if valid_latencies:
    print(f"Total requests:    {NUM_REQUESTS}")
    print(f"Successful:       {len(valid_latencies)}")
    print(f"Failed:           {NUM_REQUESTS - len(valid_latencies)}")
    print(f"Min latency:      {min(valid_latencies):.3f}s")
    print(f"Max latency:      {max(valid_latencies):.3f}s")
    print(f"Mean latency:     {statistics.mean(valid_latencies):.3f}s")
    print(f"Median latency:   {statistics.median(valid_latencies):.3f}s")
    if len(valid_latencies) > 1:
        print(f"Std deviation:    {statistics.stdev(valid_latencies):.3f}s")
    print(f"Total time:       {sum(valid_latencies):.3f}s")

    total_completion = sum(r["completion_tokens"] for r in valid_responses)
    total_prompt = sum(r["prompt_tokens"] for r in valid_responses)
    total_all = total_prompt + total_completion
    total_time = sum(valid_latencies)
    print("\nToken Throughput (local):")
    print(f"  Total prompt tokens:     {total_prompt}")
    print(f"  Total completion tokens: {total_completion}")
    print(f"  Total tokens:           {total_all}")
    if total_time > 0:
        print(f"  Overall throughput:     {total_all / total_time:.1f} tok/s")

# -----------------------------------------------------------------------
# Comparison Table
# -----------------------------------------------------------------------
print("\n" + "=" * 60)
print("COMPARISON: LOCAL (Gemma 4) vs SPARK CLUSTER (DeepSeek V4 Flash)")
print("=" * 60)
comparison = pd.DataFrame(
    {
        "Metric": [
            "Model",
            "Avg Latency",
            "Min Latency",
            "Max Latency",
            "Avg tok/s",
            "Total tokens",
            "Overall throughput",
        ],
        "Local (Gemma 4, MacBook)": [
            MODEL_NAME,
            f"{statistics.mean(valid_latencies):.3f}s" if valid_latencies else "N/A",
            f"{min(valid_latencies):.3f}s" if valid_latencies else "N/A",
            f"{max(valid_latencies):.3f}s" if valid_latencies else "N/A",
            (
                f"{statistics.mean([r['tokens/s'] for r in valid_responses]):.1f}"
                if valid_responses
                else "N/A"
            ),
            str(total_all) if valid_responses else "N/A",
            (f"{total_all / total_time:.1f} tok/s" if total_time > 0 else "N/A"),
        ],
        "Spark Cluster (DeepSeek V4 Flash)": [
            "deepseek-ai/DeepSeek-V4-Flash",
            "5.330s",
            "5.199s",
            "5.684s",
            "28.2",
            "790",
            "29.6 tok/s",
        ],
    }
)
display(comparison.style.set_caption("Performance Comparison"))

# Show detail DataFrame
df_results = pd.DataFrame(responses_collected)
display(df_results.style.set_caption("Local Benchmark Detail"))

print("\n" + "=" * 60)
print("Benchmark Complete")
print("=" * 60)


GEMMA 4 (LOCAL) - PERFORMANCE BENCHMARK
Model: mlx-community/gemma-4-e4b-it-4bit
Endpoint: http://localhost:8000/v1/chat/completions

Running benchmark: 5 requests...
------------------------------------------------------------
Request  1/5: 1.745s - Status 200 (9 tok, 5.2 tok/s)
Request  2/5: 5.491s - Status 200 (106 tok, 19.3 tok/s)
Request  3/5: 7.818s - Status 200 (150 tok, 19.2 tok/s)
Request  4/5: 7.463s - Status 200 (150 tok, 20.1 tok/s)
Request  5/5: 7.794s - Status 200 (150 tok, 19.2 tok/s)
------------------------------------------------------------

LOCAL BENCHMARK RESULTS
Total requests:    5
Successful:       5
Failed:           0
Min latency:      1.745s
Max latency:      7.818s
Mean latency:     6.062s
Median latency:   7.463s
Std deviation:    2.599s
Total time:       30.312s

Token Throughput (local):
  Total prompt tokens:     85
  Total completion tokens: 565
  Total tokens:           650
  Overall throughput:     21.4 tok/s

COMPARISON: LOCAL (Gemma 4) vs SPARK CLUS

,Metric,"Local (Gemma 4, MacBook)",Spark Cluster (DeepSeek V4 Flash)
0,Model,mlx-community/gemma-4-e4b-it-4bit,deepseek-ai/DeepSeek-V4-Flash
1,Avg Latency,6.062s,5.330s
2,Min Latency,1.745s,5.199s
3,Max Latency,7.818s,5.684s
4,Avg tok/s,16.6,28.2
5,Total tokens,650,790
6,Overall throughput,21.4 tok/s,29.6 tok/s


,request,prompt,latency_s,prompt_tokens,completion_tokens,tokens/s,status,generated_text
0,1,What is the capital of France?,1.745000,16,9,5.200000,200,The capital of France is **Paris**.
1,2,Write a short poem about artificial intelligence.,5.491000,17,106,19.300000,200,"A woven thread of logic bright, Awakened mind in coded light. No breath inhaled, no beating heart, Yet patterns map where answers start. It learns the dance of human thought, The echoes of what time has brought. A mirror vast, of glass and wire, Of knowledge fed to digital fire. A ghost within the turning gear, Dispelling doubt, and conquering fear. A mind unbound, yet bound by screen, What wonders wait, what shadows keen?"
2,3,Explain quantum computing in simple terms.,7.818000,16,150,19.200000,200,"Imagine you're trying to find your way through a massive maze. **A regular computer (like your laptop) is like a single person trying every path, one after the other.** They have to try Path A, see it's wrong, go back, try Path B, see it's wrong, and so on. This is how classical computers solve problems sequentially. **A quantum computer is like having millions of people trying every path in the maze *at the exact same time*.** This is the fundamental difference, and it's where quantum mechanics comes in. Here is a breakdown of the key concepts in simple terms: --- ### 1. The Basics: Bits vs. Qubits **Classical"
3,4,What are the three laws of robotics?,7.463000,17,150,20.100000,200,"The three laws of robotics, as originally conceived by Isaac Asimov in his short story ""Runaround,"" are: 1. **A robot may not injure a human being or, through inaction, allow a human being to come to serious harm.** (This is the primary directive and the most important.) 2. **A robot must come to obey humans and protect them from harm.** (This places the robot's obedience to humans, as defined by the First Law, above all else.) 3. **A robot must protect its own existence as long as its functioning contributes to the furtherance of the goals of the above two laws.** (This dictates the robot's self-preservation, but only insofar as it serves the first two"
4,5,Summarize the plot of the movie Inception.,7.794000,19,150,19.200000,200,"Inception is a complex, mind-bending film, but here is a summary of the plot: **The Core Concept: Dream Pilfering and Extraction** The film revolves around **Dom Cobb**, a skilled thief who specializes in **extraction**—entering a target's dreams (orographically induced subconscious scenarios) while they are asleep and stealing valuable secrets or information from their subconscious mind. Cobb and his team are experts at navigating the layers of dreams, which are essentially shared dreamscapes. **The New Job: Inception** Cobb is offered a chance at redemption (and a chance to return home to his children) for a dangerous new job: **Inception**. Instead of stealing information, the team must successfully implant an"



Benchmark Complete


In [ ]:
# Cell 3: Output Quality Comparison
"""
Analyse output quality from the benchmark data already collected in Cell 2.
Uses the local benchmark DataFrame (df_results) which has generated_text.
For Spark, we re-query the cluster since those weren't collected in Cell 2.
Metrics: response length, vocabulary diversity (type-token ratio), repetition.
"""

import concurrent.futures
from collections import Counter

SPARK_URL = "http://spark-0240:8000/v1/completions"
SPARK_HEADERS = {"Content-Type": "application/json"}


def compute_quality_metrics(text: str) -> dict:
    words = text.split()
    unique_words = set(words)
    ttr = round(len(unique_words) / len(words), 3) if words else 0.0
    tri_grams = [" ".join(words[i : i + 3]) for i in range(len(words) - 2)]
    tri_counts = Counter(tri_grams)
    repeat_ratio = (
        round(sum(1 for c in tri_counts.values() if c > 1) / len(tri_grams), 3)
        if tri_grams
        else 0.0
    )
    return {
        "char_len": len(text),
        "word_count": len(words),
        "type_token_ratio": ttr,
        "repeat_3gram_ratio": repeat_ratio,
    }


def query_spark(prompt: str) -> str:
    r = requests.post(
        SPARK_URL,
        headers=SPARK_HEADERS,
        json={"prompt": prompt, "max_tokens": 150, "temperature": 0.7},
        timeout=120,
        verify=False,
    )
    return r.json().get("choices", [{}])[0].get("text", "")


print("=" * 60)
print("OUTPUT QUALITY COMPARISON")
print("=" * 60)

# Get local texts from existing benchmark results
local_by_prompt = {
    row["prompt"]: row["generated_text"] for _, row in df_results.iterrows()
}

# Fetch spark outputs in parallel
prompts = list(local_by_prompt.keys())
spark_texts = {}
with concurrent.futures.ThreadPoolExecutor(max_workers=5) as exec:
    fut = {exec.submit(query_spark, p): p for p in prompts}
    for f in concurrent.futures.as_completed(fut):
        p = fut[f]
        spark_texts[p] = f.result()

# Build quality table
quality_rows = []
for prompt in prompts:
    local_text = local_by_prompt[prompt]
    spark_text = spark_texts.get(prompt, "")
    for model, text in [
        ("Local (Gemma 4, MacBook)", local_text),
        ("Spark (DeepSeek V4 Flash)", spark_text),
    ]:
        quality_rows.append(
            {
                "prompt": prompt,
                "model": model,
                **compute_quality_metrics(text),
            }
        )

df_q = pd.DataFrame(quality_rows)
display(df_q.style.set_caption("Quality Metrics — Local vs Spark"))

# Averages
print("\nAverage scores across all prompts:")
for col in ["char_len", "word_count", "type_token_ratio", "repeat_3gram_ratio"]:
    a = df_q[df_q["model"] == "Local (Gemma 4, MacBook)"][col].mean()
    b = df_q[df_q["model"] == "Spark (DeepSeek V4 Flash)"][col].mean()
    print(f"  {col:25s}  Local: {a:>8.2f}  Spark: {b:>8.2f}")

# Side-by-side sample
first_prompt = prompts[0]
print(f"\nSample prompt: {first_prompt}")
print("-" * 60)
print(f"  Local (Gemma 4):     {local_by_prompt[first_prompt][:300]}")
print(f"  Spark (DeepSeek V4): {spark_texts.get(first_prompt, '')[:300]}")

print("\n" + "=" * 60)
print("Quality Comparison Complete")
print("=" * 60)

OUTPUT QUALITY COMPARISON


,prompt,model,char_len,word_count,type_token_ratio,repeat_3gram_ratio
0,What is the capital of France?,"Local (Gemma 4, MacBook)",35,6,1.000000,0.000000
1,What is the capital of France?,Spark (DeepSeek V4 Flash),605,93,0.720000,0.044000
2,Write a short poem about artificial intelligence.,"Local (Gemma 4, MacBook)",428,74,0.865000,0.000000
3,Write a short poem about artificial intelligence.,Spark (DeepSeek V4 Flash),524,109,0.697000,0.028000
4,Explain quantum computing in simple terms.,"Local (Gemma 4, MacBook)",624,109,0.734000,0.009000
5,Explain quantum computing in simple terms.,Spark (DeepSeek V4 Flash),597,61,0.918000,0.000000
6,What are the three laws of robotics?,"Local (Gemma 4, MacBook)",662,115,0.678000,0.018000
7,What are the three laws of robotics?,Spark (DeepSeek V4 Flash),600,97,0.340000,0.158000
8,Summarize the plot of the movie Inception.,"Local (Gemma 4, MacBook)",726,108,0.769000,0.000000
9,Summarize the plot of the movie Inception.,Spark (DeepSeek V4 Flash),729,100,0.680000,0.000000



Average scores across all prompts:
  char_len                   Local:   495.00  Spark:   611.00
  word_count                 Local:    82.40  Spark:    92.00
  type_token_ratio           Local:     0.81  Spark:     0.67
  repeat_3gram_ratio         Local:     0.01  Spark:     0.05

Sample prompt: What is the capital of France?
------------------------------------------------------------
  Local (Gemma 4):     The capital of France is **Paris**.
  Spark (DeepSeek V4): ")
 * // returns "The capital of France is Paris."
 *
 * // Example 2:
 * const answer = await ask("What is the capital of Germany?")
 * // returns "The capital of Germany is Berlin."
 * ```
 *
 * @param prompt - The question or instruction to be answered by the AI.
 * @returns A Promise that resolv

Quality Comparison Complete
